In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from PIL import Image
from umap import UMAP
import re
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


In [ ]:
# # Load dataset
# df = pd.read_excel("3 year 21-22-23 SAC Agg.xlsx")

# df.head()
# df.columns
# df.info()

In [ ]:
# Load dataset
intakes = pd.read_csv("Austin_Animal_Center_Intakes__10_01_2013_to_05_05_2025_.csv")

intakes.head()
intakes.columns
intakes.info()

In [ ]:
# Load dataset
outcomes = pd.read_csv("Austin_Animal_Center_Outcomes__10_01_2013_to_05_05_2025_.csv")

# outcomes.head()
# outcomes.columns
# outcomes.info()

df = intakes.merge(outcomes, on="Animal ID", suffixes=("_intake", "_outcome"))

In [ ]:
def convert_age(age_str):
    if pd.isna(age_str):
        return None
    num, unit = age_str.split()
    num = int(num)
    
    if "day" in unit:
        return num
    elif "week" in unit:
        return num * 7
    elif "month" in unit:
        return num * 30
    elif "year" in unit:
        return num * 365

df["Age_days"] = df["Age upon Intake"].apply(convert_age)

In [ ]:
# Sex
df["Sex"] = df["Sex upon Intake"].str.split().str[0]  # Male/Female/Unknown

# Breed (basic grouping)
df["Breed_simple"] = df["Breed_intake"].apply(lambda x: x.split("/")[0])

# Color (take primary color)
df["Color_simple"] = df["Color_intake"].apply(lambda x: x.split("/")[0])

In [ ]:
features = df[["Age_days", "Animal Type_intake", "Sex", "Breed_simple", "Color_simple"]]

features = features.dropna()

encoder = OneHotEncoder(sparse_output=False)
X_cat = encoder.fit_transform(features.drop(columns=["Age_days"]))

import numpy as np
X = np.hstack([features[["Age_days"]].values, X_cat])

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
df_clustered = features.copy()
df_clustered["cluster"] = kmeans.fit_predict(X)

In [ ]:
cluster_age = df_clustered.groupby("cluster")["Age_days"].mean()
print(cluster_age)
sorted_clusters = cluster_age.sort_values().index

In [ ]:
cluster_map = {old: new for new, old in enumerate(sorted_clusters, start=1)}
print(cluster_map)

In [ ]:
df_clustered["cluster_ordered"] = df_clustered["cluster"].map(cluster_map)

In [ ]:
df_clustered["Outcome Type"] = df.loc[features.index, "Outcome Type"]

# Normalize distribution
cluster_outcomes = pd.crosstab(
    df_clustered["cluster_ordered"],
    df_clustered["Outcome Type"],
    normalize="index"
)

print(cluster_outcomes)

In [ ]:
df_clustered.groupby("cluster_ordered").mean(numeric_only=True)

In [ ]:
pd.crosstab(df_clustered["cluster_ordered"], df_clustered["Animal Type_intake"])

In [ ]:
df_ae = df_clustered[
    df_clustered["Outcome Type"].isin(["Adoption", "Euthanasia"])
].copy()

In [ ]:
cluster_ae = pd.crosstab(
    df_ae["cluster_ordered"],
    df_ae["Outcome Type"],
    normalize="index"
)

# Colorblind-friendly palette (Okabe-Ito)
colors = ["#0072B2","#D55E00"]

ax = cluster_ae[["Adoption","Euthanasia"]].plot(
    kind="bar",
    stacked=True,
    color=colors,
    figsize=(10,6)
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", label_type="center")

plt.title("Adoption vs Euthanasia by Age-Based Cluster")
plt.ylabel("Proportion")
plt.xlabel("Cluster (Young ? Old)")
plt.legend(title="Outcome", bbox_to_anchor=(1.05, 1))
plt.tight_layout()

labels = {
    1: "Very Young",
    2: "Young",
    3: "Adult",
    4: "Senior"
}

ax.index = cluster_ae.index.map(labels)

plt.savefig(
    f"Charts/cluster_bar_age.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
sns.boxplot(
    data=df_ae,
    x="Outcome Type",
    y="Age_days",
    palette=["#0072B2", "#D55E00"]
)

plt.title("Age Distribution: Adoption vs Euthanasia")
plt.show()

In [ ]:
cluster_ae["Euthanasia"].sort_values(ascending=False)

In [ ]:
sns.violinplot(
    data=df_ae,
    x="Outcome Type",
    y="Age_days",
    palette=["#0072B2", "#D55E00"],
    cut=0
)
plt.yscale("log")
plt.title("Age Distribution: Adoption vs Euthanasia")

plt.savefig(
    f"Charts/cluster_violin_age.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
sns.heatmap(
    cluster_outcomes,
    annot=True,
    cmap="cividis",  # VERY good for colorblindness
    fmt=".2f"
)

plt.title("Cluster vs Outcome Proportions")
plt.show()

In [ ]:
cluster_labels = {
    0: "Young Cats",
    1: "Older Dogs",
    2: "Mixed Breed Adults",
}

df_clustered["cluster_label"] = df_clustered["cluster"].map(cluster_labels)

In [ ]:
#### Colors 

df_ae = df_clustered[
    df_clustered["Outcome Type"].isin(["Adoption", "Euthanasia"])
].copy()

top_colors = df_ae["Color_simple"].value_counts().nlargest(8).index
df_color = df_ae[df_ae["Color_simple"].isin(top_colors)]

color_outcomes = pd.crosstab(
    df_color["Color_simple"],
    df_color["Outcome Type"],
    normalize="index"
).sort_index()

In [ ]:
colors = ["#0072B2", "#D55E00"]  # Adoption, Euthanasia

ax = color_outcomes[["Adoption", "Euthanasia"]].plot(
    kind="barh",
    stacked=True,
    color=colors,
    figsize=(10,6)
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", label_type="center")

plt.title("Adoption vs Euthanasia by Color")
plt.xlabel("Proportion")
plt.ylabel("Color")
plt.legend(title="Outcome")
plt.tight_layout()

plt.savefig(
    f"Charts/cluster_color.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
color_vectors = color_outcomes.copy()

kmeans = KMeans(n_clusters=3, random_state=42)
color_clusters = kmeans.fit_predict(color_vectors)

color_outcomes["color_cluster"] = color_clusters

In [ ]:
color_cluster_summary = color_outcomes.groupby("color_cluster")[["Adoption", "Euthanasia"]].mean()

color_cluster_summary.plot(
    kind="bar",
    stacked=True,
    color=colors,
    figsize=(8,5)
)

plt.title("Grouped Color Clusters vs Outcomes")
plt.show()

In [ ]:
sns.heatmap(
    color_cluster,
    cmap="cividis",
    annot=True,
    fmt=".2f"
)

plt.title("Color vs Cluster (Proportions)")
plt.xlabel("Color")
plt.ylabel("Cluster (Young ? Old)")
plt.show()

In [ ]:
color_outcome_cluster = pd.crosstab(
    [df_color["cluster_ordered"], df_color["Color_simple"]],
    df_color["Outcome Type"],
    normalize="index"
)

In [ ]:
# ##### Old, don't use 
# outcomes["DateTime"] = pd.to_datetime(outcomes["DateTime"], errors="coerce")

# adopted = outcomes[outcomes["Outcome Type"] == "Adoption"].copy()
# euthanized = outcomes[outcomes["Outcome Type"] == "Euthanasia"].copy()

# adopted["label"] = "Adopted"
# euthanized["label"] = "Euthanized"

# df_ml = pd.concat([adopted, euthanized])

In [ ]:
# features = df_ml[[
#     "Animal Type",
#     "Sex upon Outcome",
#     "Age upon Outcome",
#     "Breed",
#     "Color"
# ]]

# labels = df_ml["label"]

In [ ]:
# def convert_age(age):
#     if pd.isnull(age):
#         return None
#     num = int(re.findall(r"\d+", age)[0])
#     if "month" in age:
#         return num / 12
#     return num

# df_ml["AgeYears"] = df_ml["Age upon Outcome"].apply(convert_age)

In [ ]:
# features = df_ml[[
#     "Animal Type",
#     "Sex upon Outcome",
#     "Breed",
#     "Color",
#     "AgeYears"
# ]]

In [ ]:
# categorical_cols = ["Animal Type", "Sex upon Outcome", "Breed", "Color"]
# numeric_cols = ["AgeYears"]

# preprocessor = ColumnTransformer(
#     transformers=[
#         ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
#         ("num", "passthrough", numeric_cols)
#     ]
# )

In [ ]:
# features = features.copy()

# features = features.fillna({
#     "Animal Type": "Unknown",
#     "Sex upon Outcome": "Unknown",
#     "Breed": "Unknown",
#     "Color": "Unknown",
#     "AgeYears": features["AgeYears"].median()
# })

In [ ]:
# kmeans = KMeans(n_clusters=3, random_state=42)

# pipeline = Pipeline(steps=[
#     ("preprocessor", preprocessor),
#     ("cluster", kmeans)
# ])

# clusters = pipeline.fit_predict(features)

# df_ml["Cluster"] = clusters

In [ ]:
# print(pd.crosstab(df_ml["Cluster"], df_ml["label"], normalize="index"))

In [ ]:
# X = pipeline.named_steps["preprocessor"].transform(features)

# umap = UMAP(n_components=42)
# components = umap.fit_transform(X)
# plt.figure(figsize=(8,6))
# centroids = np.array([components[df_ml["Cluster"] == i].mean(axis=0) for i in range(3)])

# plt.scatter(
#     components[:,0],
#     components[:,1],
#     c=df_ml["Cluster"],
#     cmap="viridis",
#     alpha=0.5
#     # centroids[:,0],
#     # centroids[:,1],
#     # c="red",
#     # s=200,
#     # marker="X"
# )

# for i in range(3):
#     plt.text(
#         centroids[i,0],
#         centroids[i,1],
#         f"Cluster {i}",
#         fontsize=12
#     )

# plt.title("Animal Clusters (PCA Projection)")
# plt.xlabel("PC1")
# plt.ylabel("PC2")

# plt.savefig(
#     f"Charts/cluster1.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()

In [ ]:
# pd.crosstab(df_ml["Cluster"], df_ml["label"], normalize="index")

In [ ]:
# for col in ["Animal Type", "Sex upon Outcome", "Breed", "Color"]:
#     print("\n", col)
#     print(df_ml.groupby("Cluster")[col].value_counts(normalize=True).unstack())

In [ ]:
# df_ml.groupby("Cluster")["AgeYears"].describe()

In [ ]:
# cluster_summary = pd.crosstab(
#     df_ml["Cluster"],
#     df_ml["Animal Type"],
#     normalize="index"
# )

# sns.heatmap(cluster_summary, annot=True, cmap="Blues")
# plt.title("Cluster Composition by Animal Type")